In [13]:
# Basic Package Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error, r2_score

# imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Non-basic package imports
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import requests

# Packages I don't understand
from fcd_torch import FCD
import rdkit
from collections import Counter
import gc
import pickle

# Add the Python_files directory to the Python path
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'Python_files'))

# Now you can import your modules
import functions_enc as f
import function_depot as fd

# Morgan Fingerpring Data Processing

In [14]:
# We are switching to the June 25 dataset
df4fp = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/MIT_LL_data4_fingerprints.csv")
print(df4fp.shape)
df4fp.head()

# Uniformity of ionization model labels
print(df4fp["Ionization_Mode"].unique())
df4fp["Ionization_Mode"] = df4fp["Ionization_Mode"].replace("'Positive'", "'positive'")
print(df4fp["Ionization_Mode"].unique())
# Remove the N/A values in Ionization_Mode
df4fp = df4fp[df4fp["Ionization_Mode"] != "'N/A'"]
print(df4fp["Ionization_Mode"].unique())

# Remove single quotes from all string columns in df4
df4fp = df4fp.applymap(lambda x: x.replace("'", "") if isinstance(x, str) else x)

# print(df4fp['Group'].nunique()) # 12
# # 12 groups
# print(df4fp['Group'].unique())
# # Now we want counts of each group
# print(df4fp['Group'].value_counts())

# This will give us the subsets with all of the relevant information
df4fp_QQpos = df4fp[df4fp['Group'] == 'Q-Orbitrap-positive'] # 2065

# Other groups with their sizes listed
# df4fp_QQneg = df4fp[df4fp['Group'] == 'Q-Orbitrap-negative'] # 1400
# df4fp_QTOFpos = df4fp[df4fp['Group'] == 'Q-TOF-positive'] # 1098
# df4fp_LTQOpos = df4fp[df4fp['Group'] == 'LTQ-Orbitrap-positive'] # 615

# All of the other are just too small to be all that useful with sizes from 12 to 286 associated spectra

# df4fp_QQpos.head()


(6121, 18)
["'positive'" "'negative'" "'Positive'" "'N/A'"]
["'positive'" "'negative'" "'N/A'"]
["'positive'" "'negative'"]


In [15]:
# # SMILES count (This is the number of true ChemNet embeddings we have)
# print(df4fp['SMILES_spectra'].nunique()) # 738 unique SMILES
# print(df4fp['SMILES_spectra'].value_counts()) # 69, 64, 58, 57, 51...

# # Morgan fingerprints count
# print(df4fp['fp'].nunique()) # 658 unique Morgan fingerprints
# print(df4fp['fp'].value_counts()) 

In [16]:
# df4fp_QQpos.head()

In [23]:
# Use the function
import function_depot as fd
df4fp_QQpos_matrix = fd.expand_fingerprints_to_matrix(df4fp_QQpos)
df4fp_QQpos_matrix = df4fp_QQpos_matrix.rename(columns={'SMILES': 'SMILES_spectra'})
df4fp_QQpos_matrix.head()

Created matrix with 2065 rows and 2048 fingerprint bits
Shape: (2065, 2049)


,SMILES_spectra,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,...,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047,bit_2048
0,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# # Save the fingerprint matrix to CSV
# df4fp_QQpos_matrix.to_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/Morgan_fp_df4_QQpos.csv', index=False)
# print("Saved df4fp_QQpos_matrix to Morgan_fp_df4_QQpos.csv")

Saved df4fp_QQpos_matrix to Morgan_fp_df4_QQpos.csv


In [25]:
data = pd.read_pickle('/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes/bin0_1_thresh_zero_df_spectra.pkl')
data.head()

,SMILES_spectra,0.05,0.15000000000000002,0.25,0.35,0.44999999999999996,0.5499999999999999,0.6499999999999999,0.7499999999999999,0.8499999999999999,...,678.8500000000859,678.9500000000859,679.0500000000859,679.1500000000859,679.250000000086,679.350000000086,679.45,index_id,Response,log_response
0,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,273.642508,5.611823
1,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,273.642508,5.611823
2,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,273.642508,5.611823
3,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,273.642508,5.611823
4,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,273.642508,5.611823


In [27]:
# Training and validation dataset split for Morgan fingerprint encoder
data = pd.read_pickle('/home/dlipsey/MITLincolnLabs/MIT_LL_data/grid_search_dataframes/bin0_1_thresh_zero_df_spectra.pkl')
df4fp_QQpos_matrix = pd.read_csv('/home/dlipsey/MITLincolnLabs/MIT_LL_data/Morgan_fp_df4_QQpos.csv')

In [28]:
data.head()

,SMILES_spectra,0.05,0.15000000000000002,0.25,0.35,0.44999999999999996,0.5499999999999999,0.6499999999999999,0.7499999999999999,0.8499999999999999,...,678.8500000000859,678.9500000000859,679.0500000000859,679.1500000000859,679.250000000086,679.350000000086,679.45,index_id,Response,log_response
0,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,273.642508,5.611823
1,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,273.642508,5.611823
2,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,273.642508,5.611823
3,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,273.642508,5.611823
4,C#CCN(C)Cc1ccccc1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,273.642508,5.611823


In [29]:
df4fp_QQpos_matrix.head()

,SMILES_spectra,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,...,bit_2039,bit_2040,bit_2041,bit_2042,bit_2043,bit_2044,bit_2045,bit_2046,bit_2047,bit_2048
0,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,C#CCN(C)Cc1ccccc1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Morgan Fingerprint RF

In [ ]:
import os
import gc
import pickle

# Load Morgan fingerprint datasets folder path
morgan_folder = "/home/dlipsey/MITLincolnLabs/MIT_LL_data/morgan_enc_outputs"

# Get all .pkl files in the folder
morgan_pkl_files = [f for f in os.listdir(morgan_folder) if f.endswith('.pkl') and f.startswith('morgan_enc_')]
morgan_dataset_names = [f.replace('.pkl', '') for f in morgan_pkl_files]

print(f"Found {len(morgan_dataset_names)} Morgan fingerprint datasets to process")

# Verify we have the right count
morgan_thresh0_datasets = [name for name in morgan_dataset_names if 'thresh_zero' in name]
morgan_thresholded_datasets = [name for name in morgan_dataset_names if 'thresh_zero' not in name]

print(f"  - Morgan datasets with thresh0 (no threshold): {len(morgan_thresh0_datasets)}")
print(f"  - Morgan datasets with thresholds applied: {len(morgan_thresholded_datasets)}")

# Load the original dataset for response mapping
df4_QQpos = pd.read_csv("/home/dlipsey/MITLincolnLabs/MIT_LL_data/df4_QQpos.csv")

# Initialize storage for Morgan results
morgan_results_r2 = []
morgan_results_percent_error = []

# Dictionary to store individual errors for histogram analysis
saved_morgan_errors = {}

# Process Morgan fingerprint datasets ONE AT A TIME (memory efficient)
for i, dataset_name in enumerate(sorted(morgan_dataset_names), 1):
    print(f"Processing {i}/{len(morgan_dataset_names)}: {dataset_name}")
    
    try:
        # Load only the current Morgan dataset
        file_path = os.path.join(morgan_folder, f"{dataset_name}.pkl")
        df = pd.read_pickle(file_path)
        
        # Add response values by merging with original dataset
        df_with_response = df.merge(df4_QQpos[['SMILES_spectra', 'Response']], on='SMILES_spectra', how='left')
        
        # Add log response
        df_with_response['log_response'] = np.log(df_with_response['Response'])
        
        # Prepare features and target (Morgan fingerprint prediction columns are the features)
        feature_cols = [col for col in df.columns if col.startswith('morgan_fp_pred_')]
        X = df_with_response[feature_cols]
        y = df_with_response['log_response']
        
        # Remove rows with NaN values
        valid_mask = ~(X.isna().any(axis=1) | y.isna())
        X_clean = X[valid_mask]
        y_clean = y[valid_mask]
        
        if len(X_clean) < 10:  # Skip if too few samples
            print(f"  Skipping {dataset_name}: Only {len(X_clean)} valid samples")
            continue
            
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X_clean, y_clean, test_size=0.5, random_state=47
        )
        
        # Train Random Forest with limited CPU usage
        rf_morgan = RandomForestRegressor(n_estimators=100, random_state=47, n_jobs=2)
        rf_morgan.fit(X_train, y_train)

        # Make predictions
        y_train_pred = rf_morgan.predict(X_train)
        y_test_pred = rf_morgan.predict(X_test)
        
        # Calculate R² metrics
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        
        # Calculate absolute percent error (undo log transform first)
        y_train_true_response = np.exp(y_train)
        y_train_pred_response = np.exp(y_train_pred)
        y_test_true_response = np.exp(y_test)
        y_test_pred_response = np.exp(y_test_pred)
        
        # Calculate individual errors for test set
        individual_errors = np.abs((y_test_pred_response - y_test_true_response) / y_test_true_response) * 100
        
        # Save individual errors for histogram analysis
        saved_morgan_errors[dataset_name] = individual_errors
        
        # Calculate median and mean absolute percent error
        train_median_percent_error = 100 * (np.median(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
        test_median_percent_error = 100 * (np.median(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))
        train_mean_percent_error = 100 * (np.mean(np.abs(y_train_pred_response - y_train_true_response) / y_train_true_response))
        test_mean_percent_error = 100 * (np.mean(np.abs(y_test_pred_response - y_test_true_response) / y_test_true_response))

        # Store results
        morgan_results_r2.append({
            'Dataset': dataset_name,
            'Train_R2': train_r2,
            'Test_R2': test_r2,
            'Samples': len(X_clean),
            'Features': len(feature_cols)
        })
        
        morgan_results_percent_error.append({
            'Dataset': dataset_name,
            'Train_Median_Percent_Error': train_median_percent_error,
            'Test_Median_Percent_Error': test_median_percent_error,
            'Train_Mean_Percent_Error': train_mean_percent_error,
            'Test_Mean_Percent_Error': test_mean_percent_error,
            'Samples': len(X_clean),
            'Features': len(feature_cols)
        })
        
        print(f"Completed: Test R² = {test_r2:.4f}, Test Median % Error = {test_median_percent_error:.1f}%")
        
    except Exception as e:
        print(f"Error processing {dataset_name}: {str(e)}")
        continue
    
    finally:
        # Always clean up memory after each dataset
        if 'df' in locals():
            del df
        if 'df_with_response' in locals():
            del df_with_response
        if 'X' in locals():
            del X, y, X_clean, y_clean
        if 'rf_morgan' in locals():
            del rf_morgan
        gc.collect()
        
        # Periodic deeper cleanup every 20 datasets
        if i % 20 == 0:
            print(f"  Deep cleanup after {i} datasets...")
            gc.collect()

# Convert results to DataFrames
df_morgan_r2_results = pd.DataFrame(morgan_results_r2)
df_morgan_percent_error_results = pd.DataFrame(morgan_results_percent_error)

print(f"\nCompleted! Processed {len(morgan_results_r2)} Morgan fingerprint datasets successfully.")
print(f"Saved individual errors for {len(saved_morgan_errors)} datasets")
print(f"Results stored in: df_morgan_r2_results, df_morgan_percent_error_results")

In [ ]:
# Create the actual heatmaps for Morgan fingerprint visualization
# First, let's extract bin size and threshold from the Morgan dataset names and add to results
def parse_morgan_dataset_name(dataset_name):
    """Extract bin size and threshold from Morgan dataset name"""
    # Remove 'morgan_enc_' prefix
    name_part = dataset_name.replace('morgan_enc_', '')
    
    # Handle thresh_zero case (no threshold)
    if 'thresh_zero' in name_part:
        # Extract bin size
        bin_part = name_part.split('_thresh_zero')[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        threshold = 0.0
    else:
        # Extract bin size and threshold
        parts = name_part.split('_thresh')
        bin_part = parts[0].replace('bin', '')
        bin_size = float(bin_part.replace('_', '.'))
        
        thresh_part = parts[1].split('_df_spectra')[0]
        threshold = float(thresh_part.replace('_', '.'))
    
    return bin_size, threshold

# Add bin_size and threshold columns to Morgan results DataFrames
for df_results in [df_morgan_r2_results, df_morgan_percent_error_results]:
    bin_sizes = []
    thresholds = []
    
    for dataset_name in df_results['Dataset']:
        bin_size, threshold = parse_morgan_dataset_name(dataset_name)
        bin_sizes.append(bin_size)
        thresholds.append(threshold)
    
    df_results['BinSize'] = bin_sizes
    df_results['Threshold'] = thresholds

# Check for and remove duplicates before creating pivot tables
print("Checking for duplicates in Morgan results...")
print(f"Original df_morgan_r2_results shape: {df_morgan_r2_results.shape}")

# Remove duplicates based on BinSize + Threshold combination (keep first occurrence)
df_morgan_r2_results = df_morgan_r2_results.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')
df_morgan_percent_error_results = df_morgan_percent_error_results.drop_duplicates(subset=['BinSize', 'Threshold'], keep='first')

print(f"After removing duplicates: {df_morgan_r2_results.shape}")

# Now create pivot tables for Morgan fingerprints
morgan_r2_pivot = df_morgan_r2_results.pivot(index='BinSize', columns='Threshold', values='Test_R2') 
morgan_median_percent_error_pivot = df_morgan_percent_error_results.pivot(index='BinSize', columns='Threshold', values='Test_Median_Percent_Error')
morgan_mean_percent_error_pivot = df_morgan_percent_error_results.pivot(index='BinSize', columns='Threshold', values='Test_Mean_Percent_Error')

# List all expected thresholds
thresholds_subset = [0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 50, 100]
bins_subset = [0.05, 0.1, 0.5, 1, 2, 5, 10, 25, 50, 100, 200, 500, 1000]

# Reindex pivot tables to show all columns, filling missing with NaN
morgan_r2_pivot = morgan_r2_pivot.reindex(columns=thresholds_subset, index=bins_subset)
morgan_median_percent_error_pivot = morgan_median_percent_error_pivot.reindex(columns=thresholds_subset, index=bins_subset)
morgan_mean_percent_error_pivot = morgan_mean_percent_error_pivot.reindex(columns=thresholds_subset, index=bins_subset)

# Also create individual larger heatmaps for Morgan fingerprints for better detail
def create_detailed_heatmap_morgan(pivot_data, metric_name, cmap, figsize=(12, 8), vmin=None, vmax=None):
    """Create a detailed heatmap for a single Morgan fingerprint metric"""
    plt.figure(figsize=figsize)
    
    # Create heatmap
    sns.heatmap(pivot_data, 
                annot=True, 
                fmt='.3f' if 'R²' in metric_name else '.1f', 
                cmap=cmap,
                square=False,
                linewidths=0.5,
                vmin=vmin,
                vmax=vmax,
                cbar_kws={'label': f'Test {metric_name}', 'shrink': 0.8})
    
    plt.title(f'Morgan Fingerprint: {metric_name} by Bin Size and Threshold', fontsize=16, fontweight='bold')
    plt.xlabel('Threshold Value', fontsize=14)
    plt.ylabel('Bin Size', fontsize=14)
    plt.gca().invert_yaxis()
    
    # Improve readability
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    
    # Add text annotation for best performance
    if 'R²' in metric_name:
        best_val = pivot_data.max().max()
        plt.text(0.02, 0.98, f'Best R²: {best_val:.4f}', 
                transform=plt.gca().transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top')
    else:
        best_val = pivot_data.min().min()
        plt.text(0.02, 0.98, f'Best {metric_name}: {best_val:.1f}%', 
                transform=plt.gca().transAxes, 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                verticalalignment='top')
    
    plt.tight_layout()
    plt.savefig(f"/home/dlipsey/MITLincolnLabs/Figures/Morgan_Fingerprint_{metric_name}_by_Bin_Size_and_Threshold")
    plt.show()

# Create detailed individual Morgan fingerprint heatmaps
print("Creating detailed Morgan fingerprint heatmaps...")

# create_detailed_heatmap_morgan(morgan_r2_pivot, 'R²', 'RdYlBu')     
create_detailed_heatmap_morgan(morgan_median_percent_error_pivot, 'Median_Absolute_Percent_Error', 'RdYlBu_r', vmin=1.0, vmax=100.0) 
create_detailed_heatmap_morgan(morgan_mean_percent_error_pivot, 'Mean_Absolute_Percent_Error', 'RdYlBu_r', vmin=1.0, vmax=100.0)